In [1]:
def get_matches_today_data():
    import statsapi
    import mlbstatsapi
    from datetime import datetime

    # Get today's schedule
    matches_today = []

    # get the proper formatted date
    mlb_date = datetime.now().strftime("%m/%d/%Y")

    # get the schedule as a dictionary for today
    schedule = statsapi.schedule(start_date=mlb_date, end_date=mlb_date)

    # iterate through each game of the schedule
    for x in schedule:

        # initialize game data dictionary
        game_data = {}
        
        # away_name
        game_data.update({'away_name': x.get('away_name')}) 
        
        # home_name
        game_data.update({'home_name': x.get('home_name')})
        
        # away_id
        game_data.update({'away_id': x.get('away_id')})

        away_team_leaders_hr = []
        # add top away team guys here
        away_leaders = statsapi.team_leader_data(x.get('away_id'), 'homeRuns', season=2025, leaderGameTypes="R", limit=10)
        for z in away_leaders:
            away_team_leaders_hr.append({'name': z[1],'homeRuns': z[2]})

        game_data.update({'away_team_leaders_hr': away_team_leaders_hr})

        home_team_leaders_hr = []
        # add top away team guys here
        home_leaders = statsapi.team_leader_data(x.get('home_id'), 'homeRuns', season=2025, leaderGameTypes="R", limit=10)
        for z in home_leaders:
            home_team_leaders_hr.append({'name': z[1],'homeRuns': z[2]})

        game_data.update({'home_team_leaders_hr': home_team_leaders_hr})

        # home_id
        game_data.update({'home_id': x.get('home_id')})
        # home_probable_pitcher
        game_data.update({'home_probable_pitcher': x.get('home_probable_pitcher')})
        # away_probable_pitcher
        game_data.update({'away_probable_pitcher': x.get('away_probable_pitcher')})

        matches_today.append(game_data)


    mlb = mlbstatsapi.Mlb()

    for x in matches_today:
  
        away_probable_pitcher = x.get('away_probable_pitcher')
        
        # Check if away_probable_pitcher is valid
        if not away_probable_pitcher:
            print(f"Warning: Missing away_probable_pitcher for game: {x}")
            continue  # Skip this game if no pitcher is available

        pitcher_ids = mlb.get_people_id(away_probable_pitcher)
        
        # Check if pitcher_ids is not empty
        if not pitcher_ids:
            print(f"Warning: No pitcher ID found for {away_probable_pitcher}")
            continue  # Skip this game if no pitcher ID is found

        pitcher_id = pitcher_ids[0]  # Safely access the first element

        BvP = []
        for y in x.get('home_team_leaders_hr', []):  # Default to an empty list if key is missing
            batter_id = mlb.get_people_id(y.get('name'))[0]

            stats = ['vsPlayer']
            group = ['hitting']
            params = {'opposingPlayerId': pitcher_id, 'season': 2025}

            try:
                stats = mlb.get_player_stats(batter_id, stats=stats, groups=group, **params)
                vs_player_total = stats['hitting']['vsplayertotal']
                for split in vs_player_total.splits:
                    p_id = mlb.get_person(pitcher_id)
                    b_id = mlb.get_person(batter_id)
                    
                    bvp_matchup = f"pitcher: {p_id.__dict__.get('fullname')} vs batter: {b_id.__dict__.get('fullname')}"
                    dict2 = {'bvp_stats': split.stat.__dict__}
                    dict2.update({'bvp_matchup': bvp_matchup})
                    dict2.update({'pitcher': p_id.__dict__.get('fullname')})
                    dict2.update({'batter': b_id.__dict__.get('fullname')})
                    BvP.append(dict2)

            except KeyError as e:
                print(f"KeyError: {e}. Skipping this player. Stats: {stats}")
            except Exception as e:
                print(f"Unexpected error: {e}. Skipping this player.")
        
        
        home_probable_pitcher = x.get('home_probable_pitcher')
        
        # Check if home_probable_pitcher is valid
        if not home_probable_pitcher:
            print(f"Warning: Missing home_probable_pitcher for game: {x}")
            continue  # Skip this game if no pitcher is available

        pitcher_ids = mlb.get_people_id(home_probable_pitcher)
        
        # Check if pitcher_ids is not empty
        if not pitcher_ids:
            print(f"Warning: No pitcher ID found for {home_probable_pitcher}")
            continue  # Skip this game if no pitcher ID is found

        pitcher_id = pitcher_ids[0]  # Safely access the first element

        for y in x.get('away_team_leaders_hr', []):  # Default to an empty list if key is missing
            batter_id = mlb.get_people_id(y.get('name'))[0]

            stats = ['vsPlayer']
            group = ['hitting']
            params = {'opposingPlayerId': pitcher_id, 'season': 2025}

            try:
                stats = mlb.get_player_stats(batter_id, stats=stats, groups=group, **params)
                vs_player_total = stats['hitting']['vsplayertotal']
                for split in vs_player_total.splits:
                    p_id = mlb.get_person(pitcher_id)
                    b_id = mlb.get_person(batter_id)
                    
                    bvp_matchup = f"pitcher: {p_id.__dict__.get('fullname')} vs batter: {b_id.__dict__.get('fullname')}"
                    dict2 = {'bvp_stats': split.stat.__dict__}
                    dict2.update({'bvp_matchup': bvp_matchup})
                    dict2.update({'pitcher': p_id.__dict__.get('fullname')})
                    dict2.update({'batter': b_id.__dict__.get('fullname')})
                    BvP.append(dict2)

            except KeyError as e:
                print(f"KeyError: {e}. Skipping this player. Stats: {stats}")
            except Exception as e:
                print(f"Unexpected error: {e}. Skipping this player.")
        
        # Add the BvP stats to the matches_today dictionary
        x.update({'BvP_stats': BvP})

    return matches_today


# import sys
# import os

# sys.stdout = open(os.devnull, 'w')

# Call your function
todays_matches = get_matches_today_data()

# # Restore output
# sys.stdout = sys.__stdout__

print(todays_matches)


https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/670223/stats
KeyError: 'hitting'. Skipping this player. Stats: {}
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/669065/stats
KeyError: 'hitting'. Skipping this player. Stats: {}
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/667472/stats
KeyError: 'hitting'. Skipping this player. Stats: {}
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/682663/stats
KeyError: 'hitting'. Skipping this player. Stats: {}
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/676572/stats
KeyError: 'hitting'. Skipping this player. Stats: {}
https://statsapi.mlb.com/api/v1/sports/1/players
https://statsapi.mlb.com/api/v1/people/672640/stats
KeyError: 'hitting'. Skipping this player. Stats: {}
https://statsapi.mlb.com/ap

In [2]:
for key, value in todays_matches[0].items():
    print(f"{key}: {value}")

away_name: Los Angeles Dodgers
home_name: Miami Marlins
away_id: 119
away_team_leaders_hr: [{'name': 'Teoscar Hernández', 'homeRuns': '9'}, {'name': 'Tommy Edman', 'homeRuns': '8'}, {'name': 'Shohei Ohtani', 'homeRuns': '8'}, {'name': 'Freddie Freeman', 'homeRuns': '6'}, {'name': 'Andy Pages', 'homeRuns': '6'}, {'name': 'Mookie Betts', 'homeRuns': '5'}, {'name': 'Enrique Hernández', 'homeRuns': '5'}, {'name': 'Will Smith', 'homeRuns': '3'}, {'name': 'Michael Conforto', 'homeRuns': '2'}, {'name': 'Max Muncy', 'homeRuns': '1'}, {'name': 'Miguel Rojas', 'homeRuns': '1'}]
home_team_leaders_hr: [{'name': 'Matt Mervis', 'homeRuns': '7'}, {'name': 'Kyle Stowers', 'homeRuns': '6'}, {'name': 'Dane Myers', 'homeRuns': '3'}, {'name': 'Agustín Ramírez', 'homeRuns': '3'}, {'name': 'Eric Wagaman', 'homeRuns': '3'}, {'name': 'Otto Lopez', 'homeRuns': '2'}, {'name': 'Griffin Conine', 'homeRuns': '1'}, {'name': 'Liam Hicks', 'homeRuns': '1'}, {'name': 'Derek Hill', 'homeRuns': '1'}, {'name': 'Connor No

In [3]:

for match in todays_matches:

    if 'BvP_stats' in match:

        print("Batter vs Pitcher Stats:")
        for bvp in match['BvP_stats']:
            print()
            # print(f"Match: AWAY: {match['away_name']} vs HOME: {match['home_name']}")
            # print(f"Away Probable Pitcher: {match['away_probable_pitcher']}")
            # print(f"Home Probable Pitcher: {match['home_probable_pitcher']}")
            away = match['away_probable_pitcher']
            home = match['home_probable_pitcher']
            batter_name = bvp['bvp_matchup']
            if bvp['pitcher'] == away:
                print(f"Away Pitcher:  {bvp['pitcher']:<17} {match['away_name']:<25}")
                beans = 'away'
            elif bvp['pitcher'] == home:
                print(f"Home Pitcher:  {bvp['pitcher']:<17} {match['home_name']:<25}")
                beans = 'home'
            if beans == 'away':
                print(f"Home  Batter:  {bvp['batter']:<17} {match['home_name']:<25}")
            elif beans == 'home':
                print(f"Away  Batter:  {bvp['batter']:<17} {match['away_name']:<25}")
            # print(f"Pitcher: {bvp['pitcher']:>10}")
            print(f" AB: {bvp['bvp_stats'].get('atbats', 'N/A'):>7}")
            print(f"  H: {bvp['bvp_stats'].get('hits', 'N/A'):>7}")
            print(f" HR: {bvp['bvp_stats'].get('homeruns', 'N/A'):>7}")
            print(f"AVG: {bvp['bvp_stats'].get('avg', 'N/A'):>7}")
            print(f"RBI: {bvp['bvp_stats'].get('rbi', 'N/A'):>7}")
            print(f"obp: {bvp['bvp_stats'].get('obp', 'N/A'):>7}")
            print(f"ops: {bvp['bvp_stats'].get('ops', 'N/A'):>7}")
            print()

    print("\n")  # Print a newline for better readability between matches


Batter vs Pitcher Stats:

Home Pitcher:  Sandy Alcantara   Miami Marlins            
Away  Batter:  Teoscar Hernández Los Angeles Dodgers      
 AB:       9
  H:       3
 HR:       0
AVG:    .333
RBI:       3
obp:    .333
ops:   1.000


Home Pitcher:  Sandy Alcantara   Miami Marlins            
Away  Batter:  Tommy Edman       Los Angeles Dodgers      
 AB:       8
  H:       3
 HR:       0
AVG:    .375
RBI:       1
obp:    .400
ops:    .775


Home Pitcher:  Sandy Alcantara   Miami Marlins            
Away  Batter:  Shohei Ohtani     Los Angeles Dodgers      
 AB:       5
  H:       1
 HR:       1
AVG:    .200
RBI:       1
obp:    .333
ops:   1.133


Home Pitcher:  Sandy Alcantara   Miami Marlins            
Away  Batter:  Freddie Freeman   Los Angeles Dodgers      
 AB:      29
  H:       9
 HR:       0
AVG:    .310
RBI:       1
obp:    .394
ops:    .773


Home Pitcher:  Sandy Alcantara   Miami Marlins            
Away  Batter:  Andy Pages        Los Angeles Dodgers      
 AB:       2

In [4]:
# WRITE BVP TO A FILE

import os
from datetime import datetime, timedelta

# Ensure the "text_output" folder exists
os.makedirs("text_output", exist_ok=True)

# Get yesterday's date
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

# File paths
bvp_file_path = "text_output/BVP.txt"
backup_file_path = f"text_output/BVP_{yesterday}.txt"

# Check if BVP.txt exists and rename it
if os.path.exists(bvp_file_path):
    os.rename(bvp_file_path, backup_file_path)

# Open the new BVP.txt file in write mode
with open(bvp_file_path, "w") as file:
    for match in todays_matches:

        if 'BvP_stats' in match:

            file.write("Batter vs Pitcher Stats:\n")
            for bvp in match['BvP_stats']:
                file.write("\n")
                away = match['away_probable_pitcher']
                home = match['home_probable_pitcher']
                batter_name = bvp['bvp_matchup']
                if bvp['pitcher'] == away:
                    file.write(f"Away Pitcher:  {bvp['pitcher']:<17} {match['away_name']:<25}\n")
                    beans = 'away'
                elif bvp['pitcher'] == home:
                    file.write(f"Home Pitcher:  {bvp['pitcher']:<17} {match['home_name']:<25}\n")
                    beans = 'home'
                if beans == 'away':
                    file.write(f"Home  Batter:  {bvp['batter']:<17} {match['home_name']:<25}\n")
                elif beans == 'home':
                    file.write(f"Away  Batter:  {bvp['batter']:<17} {match['away_name']:<25}\n")
                file.write(f" AB: {bvp['bvp_stats'].get('atbats', 'N/A'):>7}\n")
                file.write(f"  H: {bvp['bvp_stats'].get('hits', 'N/A'):>7}\n")
                file.write(f" HR: {bvp['bvp_stats'].get('homeruns', 'N/A'):>7}\n")
                file.write(f"AVG: {bvp['bvp_stats'].get('avg', 'N/A'):>7}\n")
                file.write(f"RBI: {bvp['bvp_stats'].get('rbi', 'N/A'):>7}\n")
                file.write(f"obp: {bvp['bvp_stats'].get('obp', 'N/A'):>7}\n")
                file.write(f"ops: {bvp['bvp_stats'].get('ops', 'N/A'):>7}\n")
                file.write("\n")

            file.write("\n")  # Write a newline for better readability between matches

print(f"New BVP file saved to {bvp_file_path}")
if os.path.exists(backup_file_path):
    print(f"Existing BVP file renamed to {backup_file_path}")

New BVP file saved to text_output/BVP.txt
Existing BVP file renamed to text_output/BVP_2025-05-04.txt


In [5]:
# Standing and schedule into text file
import statsapi
from datetime import datetime, timedelta
import os

# Ensure the "text_output" folder exists
os.makedirs("text_output", exist_ok=True)

# Get yesterday's date
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

# File paths
file_name = "Todays_Report.txt"
report_file_path = f"text_output/{file_name}"
backup_file_path = f"text_output/Old_Report_{yesterday}.txt"

# Check if Todays_Report.txt exists and rename it
if os.path.exists(report_file_path):
    os.rename(report_file_path, backup_file_path)

# Get yesterday's schedule
oneday = timedelta(days=1)
yesterday_date = datetime.now().date() - oneday
yschedule = statsapi.schedule(start_date=yesterday_date, end_date=yesterday_date)

# Get today's schedule
mlb_date = datetime.now().strftime("%m/%d/%Y")
schedule = statsapi.schedule(start_date=mlb_date, end_date=mlb_date)

# Prepare content to write
yesterday_schedule_content = "\nYesterday's Schedule:\n" + "\n".join(
    f'{x.get("summary")}\n\n{statsapi.linescore(x.get("game_id"))}\n\n{statsapi.game_scoring_plays(x.get("game_id"))}\n\n-*-*-*-*-*-*-*-*-*-*-*-*-*-*-*\n' for x in yschedule
)

standings_content = (
    "MLB Standings:\n"
    + statsapi.standings(leagueId=103, date=mlb_date)
    + statsapi.standings(leagueId=104, date=mlb_date)
)
today_schedule_content = "Today's Schedule:\n" + "\n".join(
    f'{x.get("summary")}' for x in schedule
)

# Combine all content
full_content = (
    yesterday_schedule_content + "\n\n" +
    standings_content + "\n\n" +
    today_schedule_content
)

# Write content to the new Todays_Report.txt file
with open(report_file_path, "w") as file:
    file.write(full_content)

print(f"New report saved to {report_file_path}")
if os.path.exists(backup_file_path):
    print(f"Existing report renamed to {backup_file_path}")

New report saved to text_output/Todays_Report.txt
Existing report renamed to text_output/Old_Report_2025-05-04.txt


In [6]:
"""
away_name: Chicago Cubs
home_name: Pittsburgh Pirates
away_id: 112
away_team_leaders_hr: [{'name': 'Carson Kelly', 'homeRuns': '7'}, {'name': 'Seiya Suzuki', 'homeRuns': '7'}, {'name': 'Kyle Tucker', 'homeRuns': '7'}, {'name': 'Pete Crow-Armstrong', 'homeRuns': '6'}, {'name': 'Michael Busch', 'homeRuns': '5'}, {'name': 'Dansby Swanson', 'homeRuns': '5'}, {'name': 'Miguel Amaya', 'homeRuns': '2'}, {'name': 'Ian Happ', 'homeRuns': '2'}, {'name': 'Matt Shaw', 'homeRuns': '1'}]
home_team_leaders_hr: [{'name': 'Oneil Cruz', 'homeRuns': '8'}, {'name': 'Andrew McCutchen', 'homeRuns': '3'}, {'name': 'Bryan Reynolds', 'homeRuns': '3'}, {'name': 'Enmanuel Valdez', 'homeRuns': '2'}, {'name': 'Joey Bart', 'homeRuns': '1'}, {'name': 'Alexander Canario', 'homeRuns': '1'}, {'name': 'Henry Davis', 'homeRuns': '1'}, {'name': 'Adam Frazier', 'homeRuns': '1'}, {'name': 'Nick Gonzales', 'homeRuns': '1'}, {'name': 'Matt Gorski', 'homeRuns': '1'}, {'name': "Ke'Bryan Hayes", 'homeRuns': '1'}]
home_id: 134
home_probable_pitcher: Paul Skenes
away_probable_pitcher: Colin Rea
BvP_stats: [{'bvp_stats': {'gamesplayed': 1, 'flyouts': None, 'groundouts': 1, 'airouts': 0, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 0, 'strikeouts': 1, 'baseonballs': 1, 'intentionalwalks': 0, 'hits': 0, 'hitbypitch': 0, 'avg': '.000', 'atbats': 2, 'obp': '.333', 'slg': '.000', 'ops': '.333', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 19, 'plateappearances': 3, 'totalbases': 0, 'rbi': 0, 'leftonbase': 2, 'sacbunts': 0, 'sacflies': 0, 'babip': '.000', 'groundoutstoairouts': '1.00', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': 'pitcher: Colin Rea vs batter: Oneil Cruz', 'pitcher': 'Colin Rea', 'batter': 'Oneil Cruz'}, {'bvp_stats': {'gamesplayed': 4, 'flyouts': None, 'groundouts': 3, 'airouts': 2, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 0, 'strikeouts': 2, 'baseonballs': 1, 'intentionalwalks': 0, 'hits': 2, 'hitbypitch': 0, 'avg': '.222', 'atbats': 9, 'obp': '.300', 'slg': '.222', 'ops': '.522', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 47, 'plateappearances': 10, 'totalbases': 2, 'rbi': 0, 'leftonbase': 5, 'sacbunts': 0, 'sacflies': 0, 'babip': '.286', 'groundoutstoairouts': '1.50', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': 'pitcher: Colin Rea vs batter: Andrew McCutchen', 'pitcher': 'Colin Rea', 'batter': 'Andrew McCutchen'}, {'bvp_stats': {'gamesplayed': 4, 'flyouts': None, 'groundouts': 2, 'airouts': 0, 'runs': None, 'doubles': 2, 'triples': 0, 'homeruns': 1, 'strikeouts': 3, 'baseonballs': 0, 'intentionalwalks': 0, 'hits': 7, 'hitbypitch': 0, 'avg': '.583', 'atbats': 12, 'obp': '.583', 'slg': '1.000', 'ops': '1.583', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 42, 'plateappearances': 12, 'totalbases': 12, 'rbi': 4, 'leftonbase': 3, 'sacbunts': 0, 'sacflies': 0, 'babip': '.750', 'groundoutstoairouts': '2.00', 'catchersinterference': 0, 'atbatsperhomerun': '12.00'}, 'bvp_matchup': 'pitcher: Colin Rea vs batter: Bryan Reynolds', 'pitcher': 'Colin Rea', 'batter': 'Bryan Reynolds'}, {'bvp_stats': {'gamesplayed': 3, 'flyouts': None, 'groundouts': 0, 'airouts': 2, 'runs': None, 'doubles': 1, 'triples': 0, 'homeruns': 0, 'strikeouts': 2, 'baseonballs': 0, 'intentionalwalks': 0, 'hits': 1, 'hitbypitch': 0, 'avg': '.200', 'atbats': 5, 'obp': '.200', 'slg': '.400', 'ops': '.600', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 21, 'plateappearances': 5, 'totalbases': 2, 'rbi': 1, 'leftonbase': 2, 'sacbunts': 0, 'sacflies': 0, 'babip': '.333', 'groundoutstoairouts': '0.00', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': 'pitcher: Colin Rea vs batter: Joey Bart', 'pitcher': 'Colin Rea', 'batter': 'Joey Bart'}, {'bvp_stats': {'gamesplayed': 1, 'flyouts': None, 'groundouts': 0, 'airouts': 0, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 1, 'strikeouts': 1, 'baseonballs': 0, 'intentionalwalks': 0, 'hits': 2, 'hitbypitch': 0, 'avg': '.667', 'atbats': 3, 'obp': '.667', 'slg': '1.667', 'ops': '2.334', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 8, 'plateappearances': 3, 'totalbases': 5, 'rbi': 2, 'leftonbase': 1, 'sacbunts': 0, 'sacflies': 0, 'babip': '1.000', 'groundoutstoairouts': '-.--', 'catchersinterference': 0, 'atbatsperhomerun': '3.00'}, 'bvp_matchup': 'pitcher: Colin Rea vs batter: Henry Davis', 'pitcher': 'Colin Rea', 'batter': 'Henry Davis'}, {'bvp_stats': {'gamesplayed': 1, 'flyouts': None, 'groundouts': 0, 'airouts': 2, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 0, 'strikeouts': 1, 'baseonballs': 0, 'intentionalwalks': 0, 'hits': 0, 'hitbypitch': 0, 'avg': '.000', 'atbats': 3, 'obp': '.000', 'slg': '.000', 'ops': '.000', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 10, 'plateappearances': 3, 'totalbases': 0, 'rbi': 0, 'leftonbase': 2, 'sacbunts': 0, 'sacflies': 0, 'babip': '.000', 'groundoutstoairouts': '0.00', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': 'pitcher: Colin Rea vs batter: Adam Frazier', 'pitcher': 'Colin Rea', 'batter': 'Adam Frazier'}, {'bvp_stats': {'gamesplayed': 3, 'flyouts': None, 'groundouts': 1, 'airouts': 3, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 1, 'strikeouts': 0, 'baseonballs': 1, 'intentionalwalks': 0, 'hits': 4, 'hitbypitch': 0, 'avg': '.500', 'atbats': 8, 'obp': '.556', 'slg': '.875', 'ops': '1.431', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 28, 'plateappearances': 9, 'totalbases': 7, 'rbi': 3, 'leftonbase': 1, 'sacbunts': 0, 'sacflies': 0, 'babip': '.429', 'groundoutstoairouts': '0.33', 'catchersinterference': 0, 'atbatsperhomerun': '8.00'}, 'bvp_matchup': 'pitcher: Colin Rea vs batter: Nick Gonzales', 'pitcher': 'Colin Rea', 'batter': 'Nick Gonzales'}, {'bvp_stats': {'gamesplayed': 1, 'flyouts': None, 'groundouts': 0, 'airouts': 1, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 0, 'strikeouts': 1, 'baseonballs': 0, 'intentionalwalks': 0, 'hits': 1, 'hitbypitch': 0, 'avg': '.333', 'atbats': 3, 'obp': '.333', 'slg': '.333', 'ops': '.666', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 14, 'plateappearances': 3, 'totalbases': 1, 'rbi': 0, 'leftonbase': 1, 'sacbunts': 0, 'sacflies': 0, 'babip': '.500', 'groundoutstoairouts': '0.00', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': "pitcher: Colin Rea vs batter: Ke'Bryan Hayes", 'pitcher': 'Colin Rea', 'batter': "Ke'Bryan Hayes"}, {'bvp_stats': {'gamesplayed': 1, 'flyouts': None, 'groundouts': 0, 'airouts': 1, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 0, 'strikeouts': 0, 'baseonballs': 0, 'intentionalwalks': 0, 'hits': 1, 'hitbypitch': 0, 'avg': '.500', 'atbats': 2, 'obp': '.500', 'slg': '.500', 'ops': '1.000', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 6, 'plateappearances': 2, 'totalbases': 1, 'rbi': 0, 'leftonbase': 0, 'sacbunts': 0, 'sacflies': 0, 'babip': '.500', 'groundoutstoairouts': '0.00', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': 'pitcher: Paul Skenes vs batter: Carson Kelly', 'pitcher': 'Paul Skenes', 'batter': 'Carson Kelly'}, {'bvp_stats': {'gamesplayed': 3, 'flyouts': None, 'groundouts': 2, 'airouts': 1, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 0, 'strikeouts': 4, 'baseonballs': 0, 'intentionalwalks': 0, 'hits': 2, 'hitbypitch': 0, 'avg': '.222', 'atbats': 9, 'obp': '.222', 'slg': '.222', 'ops': '.444', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 37, 'plateappearances': 9, 'totalbases': 2, 'rbi': 0, 'leftonbase': 6, 'sacbunts': 0, 'sacflies': 0, 'babip': '.400', 'groundoutstoairouts': '2.00', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': 'pitcher: Paul Skenes vs batter: Seiya Suzuki', 'pitcher': 'Paul Skenes', 'batter': 'Seiya Suzuki'}, {'bvp_stats': {'gamesplayed': 3, 'flyouts': None, 'groundouts': 2, 'airouts': 0, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 0, 'strikeouts': 2, 'baseonballs': 0, 'intentionalwalks': 0, 'hits': 2, 'hitbypitch': 0, 'avg': '.333', 'atbats': 6, 'obp': '.333', 'slg': '.333', 'ops': '.666', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 21, 'plateappearances': 6, 'totalbases': 2, 'rbi': 1, 'leftonbase': 3, 'sacbunts': 0, 'sacflies': 0, 'babip': '.500', 'groundoutstoairouts': '2.00', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': 'pitcher: Paul Skenes vs batter: Pete Crow-Armstrong', 'pitcher': 'Paul Skenes', 'batter': 'Pete Crow-Armstrong'}, {'bvp_stats': {'gamesplayed': 4, 'flyouts': None, 'groundouts': 2, 'airouts': 0, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 0, 'strikeouts': 5, 'baseonballs': 3, 'intentionalwalks': 0, 'hits': 0, 'hitbypitch': 0, 'avg': '.000', 'atbats': 7, 'obp': '.300', 'slg': '.000', 'ops': '.300', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 53, 'plateappearances': 10, 'totalbases': 0, 'rbi': 0, 'leftonbase': 3, 'sacbunts': 0, 'sacflies': 0, 'babip': '.000', 'groundoutstoairouts': '2.00', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': 'pitcher: Paul Skenes vs batter: Michael Busch', 'pitcher': 'Paul Skenes', 'batter': 'Michael Busch'}, {'bvp_stats': {'gamesplayed': 2, 'flyouts': None, 'groundouts': 0, 'airouts': 0, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 0, 'strikeouts': 2, 'baseonballs': 1, 'intentionalwalks': 0, 'hits': 1, 'hitbypitch': 0, 'avg': '.333', 'atbats': 3, 'obp': '.500', 'slg': '.333', 'ops': '.833', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 18, 'plateappearances': 4, 'totalbases': 1, 'rbi': 0, 'leftonbase': 1, 'sacbunts': 0, 'sacflies': 0, 'babip': '1.000', 'groundoutstoairouts': '-.--', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': 'pitcher: Paul Skenes vs batter: Dansby Swanson', 'pitcher': 'Paul Skenes', 'batter': 'Dansby Swanson'}, {'bvp_stats': {'gamesplayed': 1, 'flyouts': None, 'groundouts': 2, 'airouts': 0, 'runs': None, 'doubles': 0, 'triples': 0, 'homeruns': 0, 'strikeouts': 0, 'baseonballs': 0, 'intentionalwalks': 0, 'hits': 0, 'hitbypitch': 0, 'avg': '.000', 'atbats': 2, 'obp': '.000', 'slg': '.000', 'ops': '.000', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 12, 'plateappearances': 2, 'totalbases': 0, 'rbi': 0, 'leftonbase': 0, 'sacbunts': 0, 'sacflies': 0, 'babip': '.000', 'groundoutstoairouts': '2.00', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': 'pitcher: Paul Skenes vs batter: Miguel Amaya', 'pitcher': 'Paul Skenes', 'batter': 'Miguel Amaya'}, {'bvp_stats': {'gamesplayed': 4, 'flyouts': None, 'groundouts': 2, 'airouts': 1, 'runs': None, 'doubles': 1, 'triples': 0, 'homeruns': 0, 'strikeouts': 5, 'baseonballs': 1, 'intentionalwalks': 0, 'hits': 1, 'hitbypitch': 0, 'avg': '.111', 'atbats': 9, 'obp': '.200', 'slg': '.222', 'ops': '.422', 'caughtstealing': None, 'stolenbases': None, 'stolenbasepercentage': None, 'groundintodoubleplay': 0, 'groundintotripleplay': 0, 'numberofpitches': 45, 'plateappearances': 10, 'totalbases': 2, 'rbi': 0, 'leftonbase': 2, 'sacbunts': 0, 'sacflies': 0, 'babip': '.250', 'groundoutstoairouts': '2.00', 'catchersinterference': 0, 'atbatsperhomerun': '-.--'}, 'bvp_matchup': 'pitcher: Paul Skenes vs batter: Ian Happ', 'pitcher': 'Paul Skenes', 'batter': 'Ian Happ'}]
"""
# import statsapi
# for y in todays_matches:
#     print(f"Match: AWAY: {y['away_name']} vs HOME: {y['home_name']}")
#     print()
#     print(f"Away Probable Pitcher: {y['away_probable_pitcher']}")
#     # get the stats for the away probable pitcher
#     beans = statsapi.player_stat_data(next(x['id'] for x in statsapi.get('sports_players',{'season':2025,'gameType':'W'})['people'] if x['fullName']==y.get('away_probable_pitcher')), 'pitching', 'career')
#     # Query the ERA stat
#     era1 = beans['stats'][0]['stats']['era']
#     print(f"    ERA: {era1}")
#     strikeouts1 = beans['stats'][0]['stats']['strikeoutsPer9Inn']
#     print(f"    Strikeouts per 9 Innings: {strikeouts1}")
#     print()
#     print(f"Home Probable Pitcher: {y['home_probable_pitcher']}")
#     # get the stats for the home probable pitcher
#     beans2 = statsapi.player_stat_data(next(x['id'] for x in statsapi.get('sports_players',{'season':2025,'gameType':'W'})['people'] if x['fullName']==y.get('home_probable_pitcher')), 'pitching', 'career')
#     era2 = beans2['stats'][0]['stats']['era']
#     print(f"    ERA: {era2}")
#     strikeouts2 = beans2['stats'][0]['stats']['strikeoutsPer9Inn']
#     print(f"    Strikeouts per 9 Innings: {strikeouts2}")
#     print()
#     # print(f"Away Team: {y['away_name']} (ID: {x['away_id']})")
#     print("Away Team Home Run Leaders:")
#     for leader in y['away_team_leaders_hr']:
#         print(f"  - {leader['name']}: {leader['homeRuns']} HR")

#     print()
#     # print(f"Home Team: {y['home_name']} (ID: {x['home_id']})")
#     print("Home Team Home Run Leaders:")
#     for leader in y['home_team_leaders_hr']:
#         print(f"  - {leader['name']}: {leader['homeRuns']} HR")
#     print()

# 
        
    

'\naway_name: Chicago Cubs\nhome_name: Pittsburgh Pirates\naway_id: 112\naway_team_leaders_hr: [{\'name\': \'Carson Kelly\', \'homeRuns\': \'7\'}, {\'name\': \'Seiya Suzuki\', \'homeRuns\': \'7\'}, {\'name\': \'Kyle Tucker\', \'homeRuns\': \'7\'}, {\'name\': \'Pete Crow-Armstrong\', \'homeRuns\': \'6\'}, {\'name\': \'Michael Busch\', \'homeRuns\': \'5\'}, {\'name\': \'Dansby Swanson\', \'homeRuns\': \'5\'}, {\'name\': \'Miguel Amaya\', \'homeRuns\': \'2\'}, {\'name\': \'Ian Happ\', \'homeRuns\': \'2\'}, {\'name\': \'Matt Shaw\', \'homeRuns\': \'1\'}]\nhome_team_leaders_hr: [{\'name\': \'Oneil Cruz\', \'homeRuns\': \'8\'}, {\'name\': \'Andrew McCutchen\', \'homeRuns\': \'3\'}, {\'name\': \'Bryan Reynolds\', \'homeRuns\': \'3\'}, {\'name\': \'Enmanuel Valdez\', \'homeRuns\': \'2\'}, {\'name\': \'Joey Bart\', \'homeRuns\': \'1\'}, {\'name\': \'Alexander Canario\', \'homeRuns\': \'1\'}, {\'name\': \'Henry Davis\', \'homeRuns\': \'1\'}, {\'name\': \'Adam Frazier\', \'homeRuns\': \'1\'}, {\'

In [7]:
import statsapi
import os
from datetime import datetime, timedelta

# Ensure the "text_output" folder exists
os.makedirs("text_output", exist_ok=True)

# Get yesterday's date
yesterday = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")

# File paths
file_name = "match_overviews.txt"
file_path = f"text_output/{file_name}"
backup_file_path = f"text_output/match_overviews_{yesterday}.txt"

# Check if match_overviews.txt exists and rename it
if os.path.exists(file_path):
    os.rename(file_path, backup_file_path)

# Open the new match_overviews.txt file in write mode
with open(file_path, "w") as file:
    for y in todays_matches:
        file.write(f"Match: AWAY: {y['away_name']} vs HOME: {y['home_name']}\n\n")
        file.write(f"Away Probable Pitcher: {y['away_probable_pitcher']}\n")
        
        # Get the stats for the away probable pitcher
        try:
            beans = statsapi.player_stat_data(
                next(x['id'] for x in statsapi.get('sports_players', {'season': 2025, 'gameType': 'W'})['people'] if x['fullName'] == y.get('away_probable_pitcher')),
                'pitching',
                'career'
            )
        except StopIteration:
            print(f"Error: Could not find player ID for away probable pitcher: {y.get('away_probable_pitcher')}")
            beans = None
        except Exception as e:
            print(f"An error occurred while fetching player stats: {e}")
            beans = None
        # Query the ERA stat
        era1 = beans['stats'][0]['stats']['era']
        file.write(f"    ERA: {era1}\n")
        strikeouts1 = beans['stats'][0]['stats']['strikeoutsPer9Inn']
        file.write(f"    Strikeouts per 9 Innings: {strikeouts1}\n\n")
        
        file.write(f"Home Probable Pitcher: {y['home_probable_pitcher']}\n")
        
        # Get the stats for the home probable pitcher
        try:
            beans2 = statsapi.player_stat_data(
                next(x['id'] for x in statsapi.get('sports_players', {'season': 2025, 'gameType': 'W'})['people'] if x['fullName'] == y.get('home_probable_pitcher')),
                'pitching',
                'career'
            )
        except StopIteration:
            print(f"Error: Could not find player ID for home probable pitcher: {y.get('home_probable_pitcher')}")
            beans2 = None
        except Exception as e:
            print(f"An error occurred while fetching player stats for home probable pitcher: {e}")
            beans2 = None
        era2 = beans2['stats'][0]['stats']['era']
        file.write(f"    ERA: {era2}\n")
        strikeouts2 = beans2['stats'][0]['stats']['strikeoutsPer9Inn']
        file.write(f"    Strikeouts per 9 Innings: {strikeouts2}\n\n")
        
        file.write("Away Team Home Run Leaders:\n")
        for leader in y['away_team_leaders_hr']:
            file.write(f"  - {leader['name']}: {leader['homeRuns']} HR\n")
        
        file.write("\nHome Team Home Run Leaders:\n")
        for leader in y['home_team_leaders_hr']:
            file.write(f"  - {leader['name']}: {leader['homeRuns']} HR\n")
        
        file.write("\n")

print(f"New match overview saved to {file_path}")
if os.path.exists(backup_file_path):
    print(f"Existing match overview renamed to {backup_file_path}")

New match overview saved to text_output/match_overviews.txt
Existing match overview renamed to text_output/match_overviews_2025-05-04.txt


In [10]:
import os

# Ensure the "docs" folder exists
os.makedirs("docs", exist_ok=True)

# File paths
parlay_banned_list_path = "text_output/parlay_banned_list.txt" 
todays_report_path = "text_output/Todays_Report.txt"
match_overview_path = "text_output/match_overviews.txt"
bvp_path = "text_output/BVP.txt"
output_html_path = "docs/index.html"

# Read the contents of the text files
with open(parlay_banned_list_path, "r") as parlay_banned_file:
    parlay_banned_list_content = parlay_banned_file.read()

with open(todays_report_path, "r") as todays_report_file:
    todays_report_content = todays_report_file.read()

with open(match_overview_path, "r") as match_overview_file:
    match_overview_content = match_overview_file.read()

with open(bvp_path, "r") as bvp_file:
    bvp_content = bvp_file.read()

# Create the HTML content
html_content = f"""
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>MLB Report</title>
</head>
<body>
    <ul>
    <li><a href='https://www.fantasyalarm.com/mlb/lineups'>BVP checker</a></li>
    <li><a href='https://www.baseball-reference.com'>baseball-reference</a></li>
    <li><a href='https://baseballsavant.mlb.com'>baseball-savant</a></li>
    <li><a href='https://www.fangraphs.com'>fangraphs</a></li>
    <li><a href='https://www.statmuse.com/mlb'>Stat muse</a></li>
    <li><a href='https://www.baseballmusings.com/cgi-bin/CurStreak.py'>Baseball Musings</a></li>
    </ul>
    <h1>MLB Report</h1>
    <h2>Parlay Banned List</h2>
    <pre>{parlay_banned_list_content}</pre>
    <h2>Today's Report</h2>
    <pre>{todays_report_content}</pre>
    <h2>MAtch Overviews</h2>
    <pre>{match_overview_content}</pre>
    <h2>Batter vs Pitcher Stats</h2>
    <pre>{bvp_content}</pre>
</body>
</html>
"""

# Write the HTML content to the output file
with open(output_html_path, "w") as output_file:
    output_file.write(html_content)

print(f"HTML file saved to {output_html_path}")

HTML file saved to docs/index.html


In [1]:
from bs4 import BeautifulSoup
import re
from collections import Counter

# File paths
input_html_path = "docs/index.html"
output_html_path = "docs/index2.html"

# Read the HTML content
with open(input_html_path, "r") as input_file:
    html_content = input_file.read()

# Parse the HTML content
soup = BeautifulSoup(html_content, "html.parser")

# Extract all text from the HTML
text_content = soup.get_text()

# Define a regex pattern to match player names (assuming names are two words with capitalized initials)
player_name_pattern = r'\b[A-Z][a-z]+\s[A-Z][a-z]+\b'

# Find all player names in the text
player_names = re.findall(player_name_pattern, text_content)

# Count occurrences of each player name
player_counts = Counter(player_names)

# Get the top 5 players
top_players = player_counts.most_common(5)

# Create a dictionary to map players to their teams
player_teams = {}
for match in todays_matches:
    for leader in match['away_team_leaders_hr']:
        player_teams[leader['name']] = match['away_name']
    for leader in match['home_team_leaders_hr']:
        player_teams[leader['name']] = match['home_name']

# Prepare the top players list with their teams
top_players_with_teams = [
    f"{player} ({player_teams.get(player, 'Unknown Team')}) - {count} mentions"
    for player, count in top_players
]

# Create the HTML for the top players list
top_players_html = "<h2>Top Mentioned Players</h2><ul>"
for player_info in top_players_with_teams:
    top_players_html += f"<li>{player_info}</li>"
top_players_html += "</ul>"

# Prepend the top players list to the original HTML
body = soup.body
body.insert(0, BeautifulSoup(top_players_html, "html.parser"))

# Write the modified HTML to the new file
with open(output_html_path, "w") as output_file:
    output_file.write(str(soup))

print(f"Updated HTML file saved to {output_html_path}")

NameError: name 'todays_matches' is not defined

Updated HTML file saved to docs/index2.html


/tmp/ipykernel_1633/2482908357.py:47: DeprecationWarning: The 'text' argument to find()-type methods is deprecated. Use 'string' instead.
  matches_section = soup.find("h2", text="Today's Schedule")
